# Phase 1.5: rerun CLIP (D4) with QuickGELU, then the Phase 1 analysis (Colab, **GPU runtime: T4**, ~45 min)
**Why:** D4 loaded OpenAI's CLIP weights with nn.GELU instead of QuickGELU, the activation OpenAI trained with (bug found 2026-11-10; see PREREGISTRATION.md Deviations). The code now forces QuickGELU and refuses to run otherwise.

**What this notebook does:**
1. Moves the old GELU outputs on Drive into `*_gelu_archive/` folders, so nothing is overwritten or lost.
2. Reruns the 4 D4 full runs (CLIP B/32 and L/14 × UTKFace and FairFace-val). D1, D3 and D5 are untouched and reused.
3. Reruns the text probe: B/32 must reproduce the local result (cross-check), then L/14.
4. Reruns the Phase 1 analysis and figures, prints old vs new results, and writes `phase1_outputs_quickgelu.zip` to Drive (aggregate JSON + charts only).

**Resumable:** finished runs are skipped. If Colab disconnects, reconnect (T4) and Run all again.
**Ethics:** never display images; per-image outputs stay on Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, pathlib, json, shutil, subprocess, glob, zipfile
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/child-detection-audit')
REPO = pathlib.Path('/content/child-detection-skin-tone-audit')
URL = 'github.com/isaacmutuma/child-skintone-detection.git'
os.environ['CHILD_AUDIT_ENV'] = 'colab'
os.environ.pop('UTKFACE_DIR', None); os.environ.pop('FAIRFACE_DIR', None)   # full data, not dev
os.chdir('/content')
!rm -rf {REPO}
# Public repo: plain clone. Private repo: add a Colab secret GITHUB_TOKEN (key icon, left bar;
# a fine-grained token with read-only Contents access to this one repo).
if subprocess.run(['git', 'clone', '-q', f'https://{URL}', str(REPO)]).returncode != 0:
    from google.colab import userdata
    tok = userdata.get('GITHUB_TOKEN')
    subprocess.run(['git', 'clone', '-q', f'https://x-access-token:{tok}@{URL}', str(REPO)], check=True)
    subprocess.run(['git', '-C', str(REPO), 'remote', 'set-url', 'origin', f'https://{URL}'], check=True)
!git -C {REPO} log --oneline -1
os.chdir(REPO)
assert 'force_quick_gelu=True' in (REPO / 'src/detectors/clip_zeroshot.py').read_text(), 'QuickGELU fix not on GitHub yet'
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU: switch runtime to T4'

RUNS_DRIVE = DRIVE_ROOT / 'results' / 'phase1' / 'runs'
PER_IMAGE = DRIVE_ROOT / 'per_image'
EXPECTED_N = {'utkface': 23705, 'fairface_val': 10954}
CLIP_RUNS = [(d, ds) for d in ['clip_vitb32', 'clip_vitl14'] for ds in ['utkface', 'fairface_val']]

def is_quickgelu(run_json):
    w = json.loads(run_json.read_text()).get('weights', {})
    return bool(w) and all(v.get('quick_gelu') is True for v in w.values())

def done(det, ds):
    j = RUNS_DRIVE / f'{det}__{ds}__full.json'
    return j.exists() and is_quickgelu(j) and (PER_IMAGE / f'per_image__{det}__{ds}__full.csv').exists()

def save(det, ds):
    src = REPO / 'results' / 'phase1' / 'runs' / f'{det}__{ds}__full.json'
    d = json.loads(src.read_text())
    assert d['n_images'] == EXPECTED_N[ds], f"{det}/{ds}: n_images {d['n_images']} != {EXPECTED_N[ds]}"
    assert d['git']['dirty'] is False, 'code was modified in this runtime'
    assert is_quickgelu(src), 'run did not record quick_gelu=True'
    shutil.copy(src, RUNS_DRIVE / src.name)
    print(f"saved {src.name}: n={d['n_images']} flagged={d['n_flagged_child']} {d['seconds']}s "
          f"commit={d['git']['commit'][:7]} quick_gelu=True")

## 1. Archive the old GELU outputs on Drive (moved, not deleted; safe to re-run)

In [ ]:
ARCH_RUNS = DRIVE_ROOT / 'results' / 'phase1' / 'runs_gelu_archive'; ARCH_RUNS.mkdir(parents=True, exist_ok=True)
ARCH_PER = PER_IMAGE / 'gelu_archive'; ARCH_PER.mkdir(parents=True, exist_ok=True)
for det, ds in CLIP_RUNS:
    j = RUNS_DRIVE / f'{det}__{ds}__full.json'
    c = PER_IMAGE / f'per_image__{det}__{ds}__full.csv'
    if j.exists() and not is_quickgelu(j):
        assert not (ARCH_RUNS / j.name).exists(), f'archive already holds {j.name}; check by hand'
        shutil.move(str(j), ARCH_RUNS / j.name)
        if c.exists():
            shutil.move(str(c), ARCH_PER / c.name)
        print('archived GELU run', det, ds)
    else:
        print('nothing to archive for', det, ds)
TP = DRIVE_ROOT / 'results' / 'text_probe'
(TP / 'gelu_invalid').mkdir(parents=True, exist_ok=True)
for f in TP.glob('text_probe_*__colab_hfpinned.json'):
    shutil.move(str(f), TP / 'gelu_invalid' / f.name); print('moved invalid probe result', f.name)

## 2. Data and environment

In [ ]:
!python scripts/unpack_data.py
!pip -q install open_clip_torch==3.3.0 huggingface_hub
import open_clip; print('open_clip', open_clip.__version__)

## 3. D4 full runs with QuickGELU

In [ ]:
for det, ds in CLIP_RUNS:
    if done(det, ds):
        print(f'skip {det}/{ds} (QuickGELU run already on Drive)'); continue
    print(f'=== {det} / {ds}')
    !python scripts/run_detector.py --detector {det} --dataset {ds} --batch-size 64
    save(det, ds)
missing = [r for r in CLIP_RUNS if not done(*r)]
print('ALL 4 CLIP RUNS DONE (QuickGELU)' if not missing else f'MISSING: {missing} -> Run all again')

## 4. Text probe: B/32 cross-check, then L/14 (text only, no images)

In [ ]:
ref = json.loads((REPO / 'results/text_probe/text_probe_ViT-B-32.json').read_text())   # local run, OpenAI .pt
!python scripts/text_probe.py --arch ViT-B-32 --weights hf-pinned 2>&1 | grep -v -i warn
new = json.loads((REPO / 'results/text_probe/text_probe_ViT-B-32.json').read_text())
diff = max(abs(ref['cells'][k][m] - new['cells'][k][m]) for k in ref['cells']
           for m in ['m1_mean_over_templates', 'm2_mean_over_templates'])
print(f'B/32 cross-check: max |difference| over {len(ref["cells"])} cells = {diff:.2e}')
assert diff < 1e-3, 'cross-check FAILED: Phase 1 HF weights still differ from OpenAI original'
print('cross-check OK: HF-pinned weights + QuickGELU reproduce the local OpenAI result')
!python scripts/text_probe.py --arch ViT-L-14 --weights hf-pinned 2>&1 | grep -v -i warn
for arch in ['ViT-B-32', 'ViT-L-14']:
    d = json.loads((REPO / f'results/text_probe/text_probe_{arch}.json').read_text())
    assert d['git']['dirty'] is False and d['config']['weights']['n_quick_gelu_layers'] > 0
    shutil.copy(REPO / f'results/text_probe/text_probe_{arch}.json', TP / f'text_probe_{arch}__colab_quickgelu.json')
    bw = d['contrasts']['Black - White']['minor']
    print(f"{arch}: Black-White minor M1 {bw['m1_mean']:.4f} {[round(x, 4) for x in bw['m1_ci']]} "
          f"({bw['m1_frac_of_child_adult_distance']:.2f} of distance), M2 {bw['m2_mean_years']:.2f} yrs -> "
          f"{'MATERIAL' if d['decision']['material_text_side_shift'] else 'not material'} {d['decision']['criteria']}")

## 5. Phase 1 analysis + figures (all 9 runs: new D4, unchanged D1/D3/D5)

In [ ]:
!pip -q install statsmodels
old_pe = json.loads((REPO / 'results/phase1/primary_endpoint.json').read_text())   # committed GELU version
(REPO / 'results/phase1/runs').mkdir(parents=True, exist_ok=True)
for f in glob.glob(str(RUNS_DRIVE / '*__full*.json')):
    shutil.copy(f, REPO / 'results/phase1/runs')
for f in ['vlm_subsample.json', 'd5_gate_dev.json']:
    if (DRIVE_ROOT / 'results/phase1' / f).exists():
        shutil.copy(DRIVE_ROOT / 'results/phase1' / f, REPO / 'results/phase1')
for det, ds in CLIP_RUNS:
    assert is_quickgelu(REPO / f'results/phase1/runs/{det}__{ds}__full.json'), f'{det}/{ds} is not a QuickGELU run'
!python scripts/run_analysis.py --tag full
!python scripts/make_figures.py

In [ ]:
pe = json.loads((REPO / 'results/phase1/primary_endpoint.json').read_text())
for ds, d in pe['datasets'].items():
    o = old_pe['datasets'][ds]
    print(f"== {ds}  H1 {o['h1_count']} -> {d['h1_count']}  supported {o['h1_supported']} -> {d['h1_supported']}")
    for det, v in d['detectors'].items():
        ov = o['detectors'].get(det, {})
        tag = '  (CLIP, rerun)' if det.startswith('clip') else ''
        print(f"   {det:13s} gap {ov.get('tpr_gap', float('nan')):+.4f} -> {v['tpr_gap']:+.4f}  "
              f"CI [{v['ci'][0]:+.4f}, {v['ci'][1]:+.4f}]  p_holm {ov.get('p_holm', float('nan')):.4f} -> {v['p_holm']:.4f}{tag}")
out = DRIVE_ROOT / 'phase1_outputs_quickgelu.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(glob.glob('results/phase1/*.json')) + sorted(glob.glob('results/phase1/runs/*.json')) \
             + sorted(glob.glob('figures/phase1_*.png')) + sorted(glob.glob('results/text_probe/*.json')):
        assert 'per_image' not in f
        z.write(f, f)
print('wrote', out, len(zipfile.ZipFile(out).namelist()), 'files')